# public smoke - does the public repository run on a T4? (about 15-20 minutes)

Clones the PUBLIC repository without any token, checks out the pinned commit (`PINNED_SHA`, refused
on any mismatch), installs from `uv.lock`, and decodes the 5 smoke documents of `splits/smoke5.json`
with the production configuration (`qwen35_4b_img_only_native`: Qwen3.5-4B, image only, native
resolution, keyed JSON under a grammar, greedy, seed 42, batch size 1). Then the 7 assertions of
the 02n smoke gate (`shipdoc.smoke`: JSON validity, no truncation, row counts, quantity and total
shares, no all-null rows, finite logprobs for every emitted field). It never scores anything and
never writes to Drive: all outputs stay under `/content`.

Needs: a T4 runtime, `MyDrive/shipdoc-extract/data.zip` (the evaluators' images and labels; only
the 5 smoke documents are read out of it) AND `MyDrive/shipdoc-extract/assignment.zip` (your evaluator
package: the schema and the scorer, which are not part of this repository; checked before the GPU
work starts). No secret, no token, no Weights & Biases.

**Parameters:** `DETERMINISM_DOCS` (default 2): that many smoke documents are decoded a second time
in a fresh process and their raw page texts must be byte-identical to the first pass (about 3
minutes extra; 0 skips it).

The last cell prints a banner (pin, python / torch / transformers / xgrammar versions, GPU, smoke
result, seconds, determinism). It holds no document value: paste it back as it is. Timing basis,
stated plainly: the smoke gate took 348.6 s on a T4 in the original v1.5 run (`manifest.json`
`timings.smoke_s`); the install and the model download are NOT MEASURED here.


In [ ]:
# Parameters. PINNED_SHA = the PUBLIC repo commit this notebook is pinned to (40-char SHA).
import json
import os
import time

PINNED_SHA = "3e88cc8aa4bc2c33ac676c4c52395271f570b9cd"
RUN_NAME = "publicsmoke"
CONFIG = "qwen35_4b_img_only_native"  # production: native resolution (2145 visual tokens per page)
SMOKE_DOCS = "splits/smoke5.json"  # 5 dev docs picked by meta tags (shipdoc.smoke)
SMOKE_LIMIT = 5
DETERMINISM_DOCS = 2  # decoded a second time in a fresh process; 0 = skip
MODE = "run"  # the shared install cell takes these three; nothing here uses W&B or a Hub token
USE_WANDB = False
HF_TOKEN = None  # Qwen/Qwen3.5-4B is public: anonymous download

if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell (the pin script fills it).")
if not (isinstance(DETERMINISM_DOCS, int) and 0 <= DETERMINISM_DOCS <= SMOKE_LIMIT):
    raise ValueError(f"DETERMINISM_DOCS must be an int in 0..{SMOKE_LIMIT}")
SHA7 = PINNED_SHA[:7]
RUN_ID = f"{RUN_NAME}_{CONFIG}_{SHA7}"
T_START = time.time()


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
for _zip in ("data.zip", "assignment.zip"):
    if not (DRIVE_DIR / _zip).is_file():
        raise FileNotFoundError(
            f"Missing MyDrive/shipdoc-extract/{_zip}: upload BOTH data.zip and assignment.zip "
            "(your evaluator package) to MyDrive/shipdoc-extract/ (the repository README says how)."
        )
RUNS_DIR = Path("/content/runs")  # local disk, fresh per session: nothing is resumed or kept
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("data.zip and assignment.zip present; outputs go to", RUNS_DIR)


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gaurav-gandhi-2411/shipping-doc-extraction.git"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
# Pin verification: the working tree IS the pinned commit (nothing modified); printed in the banner.
dirty = git("status", "--porcelain", "--untracked-files=no")
assert not dirty, f"the clone differs from the pinned commit: {dirty[:200]}"
TREE_SHA = git("rev-parse", "HEAD^{tree}")
assert (REPO / "uv.lock").is_file() and (REPO / "splits" / "smoke5.json").is_file(), "not the repo"
print(f"pin OK: HEAD {head}, tree {TREE_SHA}")


In [ ]:
# Only the smoke documents are read out of data.zip: dev page images and dev labels (the gate
# compares the emitted row counts with the labels' row counts; no label value is printed).
DATA_DIR = Path("/content/data")
ASSIGNMENT_DIR = Path("/content/assignment")  # the install cell points SHIPDOC_ASSIGNMENT_DIR here
DEV_LABELS = DATA_DIR / "dev" / "labels"
smoke_ids = json.loads((REPO / SMOKE_DOCS).read_text(encoding="utf-8"))
assert len(smoke_ids) == 5 == len(set(smoke_ids)), f"{SMOKE_DOCS}: expected 5 distinct documents"
local_zip = Path("/content/data.zip")
shutil.copyfile(DRIVE_DIR / "data.zip", local_zip)  # local disk: Drive reads are slow
wanted = tuple(
    prefix for d in smoke_ids for prefix in (f"data/dev/images/{d}_p", f"data/dev/labels/{d}.json")
)
with zipfile.ZipFile(local_zip) as zf:
    members = [n for n in zf.namelist() if n.startswith(wanted)]
    zf.extractall("/content", members)
for d in smoke_ids:
    assert (DEV_LABELS / f"{d}.json").is_file(), f"data.zip has no label file for {d}"
    assert list((DATA_DIR / "dev" / "images").glob(f"{d}_p*")), f"data.zip has no images for {d}"
print(f"smoke data OK: {len(smoke_ids)} documents, {len(members)} files extracted")

# assignment.zip (the evaluators' package, never vendored in this repository) holds assignment/ with
# schema.json (the post-processing rules read it) and score.py. Checked BEFORE the install and the
# model cells so that a missing or wrong package costs seconds, not a GPU run.
shutil.copyfile(DRIVE_DIR / "assignment.zip", Path("/content/assignment.zip"))
with zipfile.ZipFile("/content/assignment.zip") as zf:
    zf.extractall("/content")
for _need in ("schema.json", "score.py"):
    assert (ASSIGNMENT_DIR / _need).is_file(), (
        f"assignment.zip has no assignment/{_need}: upload BOTH data.zip and assignment.zip "
        "(your evaluator package) to MyDrive/shipdoc-extract/"
    )
print("assignment OK: schema.json and score.py present in", ASSIGNMENT_DIR)


In [ ]:
import collections
import subprocess

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(DATA_DIR),
    "SHIPDOC_ASSIGNMENT_DIR": str(ASSIGNMENT_DIR),
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",  # fewer fragmentation OOMs
    "SHIPDOC_OCR_CACHE": "/content/ocr_cache",  # unused by img_only; never read
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
}
if USE_WANDB:
    ENV.update(WANDB_API_KEY=WANDB_API_KEY, WANDB_PROJECT=WANDB_PROJECT, WANDB_RUN_GROUP=RUN_NAME)
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
VLM_GROUP = ["--group", "vlm"] if MODE == "run" else []  # merge mode needs no torch / model stack
rc, _ = run_stream(["uv", "sync", "--frozen", *VLM_GROUP, "--python", "3.11"])
assert rc == 0, f"uv sync failed (exit {rc})"

if MODE == "run":
    subprocess.run(["nvidia-smi"], check=False)
    PROBE = (
        "import json, importlib.metadata as m, torch;"
        "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
        " 'arch': torch.cuda.get_arch_list(),"
        " **{p: m.version(p) for p in ('transformers', 'xgrammar')}}))"
    )
    info = json.loads(subprocess.run([PY, "-c", PROBE], env=ENV, capture_output=True, text=True,
                                     check=True).stdout.strip().splitlines()[-1])
    print(info)
    assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> T4 GPU."
    # T4 is sm_75: need a native sm_75 kernel or PTX (compute_XX with XX <= 75) it can JIT.
    ok = "sm_75" in info["arch"] or any(
        a.startswith("compute_") and int(a.split("_")[1]) <= 75 for a in info["arch"]
    )
    assert ok, f"torch build has no sm_75 support: {info['arch']}"


In [ ]:
# MANDATORY smoke gate (keyed format, prompt v2, logprobs: unverified on a GPU):
# the keyed config on the 5 smoke docs, then 7 assertions on the output (shipdoc.smoke):
# JSON validity, no truncation, row counts, quantity / total share, no all-null rows, and finite
# field logprobs for every emitted field. A failure raises: the full run below is NOT started.
# Resumable via a status file on Drive (a passed smoke for this run id is not repeated).
import importlib.util
import sys
import time

import yaml

_smoke_path = REPO / "src" / "shipdoc" / "smoke.py"
_spec = importlib.util.spec_from_file_location("shipdoc_smoke", _smoke_path)
smoke = importlib.util.module_from_spec(_spec)
sys.modules["shipdoc_smoke"] = smoke  # dataclasses resolves annotations through sys.modules
_spec.loader.exec_module(smoke)

SMOKE_RUNS = RUNS_DIR / "smoke"
SMOKE_RUN_ID = f"smoke_{RUN_ID}"
SMOKE_STATUS_PATH = RUNS_DIR / f"{RUN_NAME}_smoke_status.json"
smoke_status = json.loads(SMOKE_STATUS_PATH.read_text()) if SMOKE_STATUS_PATH.is_file() else {}
SMOKE_ENV = {**ENV, "SHIPDOC_RUNS_DIR": str(SMOKE_RUNS)}
for _k in ("WANDB_API_KEY", "WANDB_PROJECT", "WANDB_RUN_GROUP"):
    SMOKE_ENV.pop(_k, None)  # smoke runs never log to W&B
SMOKE_PASSED = False


def save_smoke_status() -> None:
    tmp = SMOKE_STATUS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(smoke_status, indent=1))
    tmp.replace(SMOKE_STATUS_PATH)  # atomic: a disconnect never leaves a half-written file


if smoke_status.get("state") == "passed" and smoke_status.get("run_id") == SMOKE_RUN_ID:
    SMOKE_PASSED = True
    print(f"SKIP smoke: already passed ({SMOKE_RUN_ID})")
else:
    run_dir = SMOKE_RUNS / SMOKE_RUN_ID
    t0 = time.time()
    rc, tail, checks, warns, error = 0, [], [], [], None
    try:
        max_new = int(yaml.safe_load((REPO / f"configs/spike_{CONFIG}.yaml").read_text())[
            "max_new_tokens"])
        if not (run_dir / "metrics.json").is_file():  # a finished run is re-checked, not re-run
            print(f"=== {SMOKE_RUN_ID}: {len(smoke_ids)} docs ===", flush=True)
            cmd = [PY, "-m", "shipdoc", "spike", "--config", f"configs/spike_{CONFIG}.yaml",
                   "--docs", SMOKE_DOCS, "--split", "dev", "--run-id", SMOKE_RUN_ID, "--resume",
                   "--limit", str(SMOKE_LIMIT), "--logprobs"]
            rc, tail = run_stream(cmd, env=SMOKE_ENV)
        if rc == 0:
            checks = smoke.load_and_check(run_dir, DEV_LABELS, max_new, require_logprobs=True)
            warns = smoke.load_warnings(run_dir)  # non-blocking: never raises, never fails
    except (OSError, KeyError, ValueError) as exc:  # fail closed: could not verify = fail
        rc, error = 1, f"{type(exc).__name__}: {exc}"
    if rc != 0:
        state = "error"
        print(f"SMOKE ERROR: exit {rc} {error or ''}; last output: {tail[-3:]}")
    else:
        state = "passed" if all(c.passed for c in checks) else "failed"
        print(smoke.format_table(CONFIG, checks, warns))
    smoke_status = {
        "state": state,
        "run_id": SMOKE_RUN_ID,
        "exit_code": rc,
        "seconds": round(time.time() - t0, 1),
        "checks": [{"name": c.name, "passed": c.passed, "detail": c.detail} for c in checks],
        "warnings": [{"name": w.name, "count": w.count, "detail": w.detail} for w in warns],
        "error": error,
        "tail": tail[-5:] if rc else [],
    }
    save_smoke_status()
    SMOKE_PASSED = state == "passed"

if not SMOKE_PASSED:
    raise RuntimeError(
        "SMOKE GATE FAILED: the full run is NOT started. GG: report back with "
        f"{SMOKE_STATUS_PATH} and the per-assertion table printed above, plus (they stay on "
        f"Drive, do not paste document values into chat) "
        f"{SMOKE_RUNS / SMOKE_RUN_ID}/trace.jsonl "
        "and predictions.json. Do not edit thresholds to make it pass."
    )
print("SMOKE GATE PASSED: starting the full run.")


In [ ]:
# DETERMINISM (cheap): the first DETERMINISM_DOCS smoke documents are decoded AGAIN in a fresh
# process (own run id, same config, seed 42, same batch size) and the raw page texts must be
# byte-identical to the smoke pass. Counts and hashes only; informational, never blocks.
import hashlib

DET = {"requested": DETERMINISM_DOCS, "ran": False, "identical": None, "seconds": None}


def page_hashes(run_dir):
    """{doc_id: [sha256 of every page's raw text]} of a run's trace (no text is kept)."""
    out = {}
    for line in (run_dir / "trace.jsonl").read_text(encoding="utf-8").splitlines():
        if line.strip():
            t = json.loads(line)
            out[t["doc_id"]] = [
                hashlib.sha256(p["raw_text"].encode()).hexdigest() for p in t["pages"]
            ]
    return out


if DETERMINISM_DOCS and SMOKE_PASSED:
    DET_ID = f"det_{RUN_ID}"
    t_det = time.time()
    det_cmd = [PY, "-m", "shipdoc", "spike", "--config", f"configs/spike_{CONFIG}.yaml",
               "--docs", SMOKE_DOCS, "--split", "dev", "--run-id", DET_ID, "--resume",
               "--limit", str(DETERMINISM_DOCS), "--logprobs"]
    rc_det, tail_det = run_stream(det_cmd, env=SMOKE_ENV)
    DET["seconds"] = round(time.time() - t_det, 1)
    if rc_det != 0:
        print(f"DETERMINISM RUN ERROR: exit {rc_det}; last output: {tail_det[-3:]}")
    else:
        first, second = page_hashes(SMOKE_RUNS / SMOKE_RUN_ID), page_hashes(SMOKE_RUNS / DET_ID)
        DET["ran"] = True
        DET["identical"] = bool(second) and all(first.get(d) == h for d, h in second.items())
        n_pages = sum(len(h) for h in second.values())
        print(f"determinism: {len(second)} documents / {n_pages} pages decoded twice, "
              f"byte-identical raw text: {DET['identical']}")
else:
    print("determinism check skipped (DETERMINISM_DOCS = 0 or the smoke gate did not pass)")


In [ ]:
# Banner to paste back. Reads files under /content only; no document value is printed.
import platform
import subprocess

bar = "=" * 78
status_path = RUNS_DIR / f"{RUN_NAME}_smoke_status.json"
status = json.loads(status_path.read_text()) if status_path.is_file() else {}


def out(cmd):
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=60, check=False)
        return res.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        return ""


gpu = (out(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
       .splitlines() or ["none"])[0]
venv_py = out([PY, "--version"])
probe = ("import importlib.metadata as m, torch; "
         "print(torch.__version__, m.version('transformers'), m.version('xgrammar'))")
torch_v, tf_v, xg_v = (out([PY, "-c", probe]).split() + ["?", "?", "?"])[:3]
state = status.get("state", "not run")
checks = status.get("checks", [])
passed = sum(1 for c in checks if c.get("passed"))
det = globals().get("DET") or {}
det_txt = "skipped"
if det.get("ran"):
    det_txt = f"identical={det['identical']} ({det['seconds']} s)"
print(bar)
print("PUBLIC SMOKE BANNER (paste this block back; it holds no document value)")
print(bar)
print(f"repository   : {REPO_URL}")
tree = globals().get("TREE_SHA", "?")[:12]
print(f"pin          : {PINNED_SHA}  (HEAD == pin asserted; tree {tree})")
print(f"python       : kernel {platform.python_version()}; project venv {venv_py}")
print(f"torch / transformers / xgrammar : {torch_v} / {tf_v} / {xg_v}")
print(f"GPU          : {gpu}")
print(f"config       : {CONFIG}")
print(f"smoke gate   : {state.upper()} - {passed}/{len(checks)} assertions passed, "
      f"{status.get('seconds')} s")
for c in checks:
    print(f"  {'PASS' if c.get('passed') else 'FAIL'}  {c.get('name')}")
print(f"determinism  : {det_txt}")
print(f"wall-clock   : {round(time.time() - T_START)} s since the parameters cell ran")
print(bar)
print(f"DONE {RUN_ID} smoke={state} determinism={det.get('identical')} pin={SHA7}")
assert state == "passed", "SMOKE GATE DID NOT PASS: paste this banner and the tables above back"
